# Synth Eval privacy evaluation

In [1]:
%pip install -U "synthcity[privacy]" 

  Using cached synthcity-0.2.12-py3-none-any.whl.metadata (37 kB)
  Using cached importlib_metadata-8.7.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached nflows-0.14-py3-none-any.whl
  Using cached lifelines-0.29.0-py3-none-any.whl.metadata (3.2 kB)
  Using cached networkx-2.8.8-py3-none-any.whl.metadata (5.1 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached xgboost-2.1.4-py3-none-manylinux_2_28_x86_64.whl.metadata (2.1 kB)
  Using cached geomloss-0.2.6-py3-none-any.whl
  Using cached pgmpy-0.1.26-py3-none-any.whl.metadata (9.1 kB)
  Using cached pycox-0.3.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached xgbse-0.3.3-py3-none-any.whl.metadata (17 kB)
  Using cached pykeops-2.3-py3-none-any.whl
  Using cached fflows-0.0.3-py3-none-any.whl.metadata (3.4 kB)
  Using cached be_great-0.0.9-py3-none-any.whl.metadata (5.8 kB)
  Using cached arfpy-0.1.1-py3-none-any.whl
  Using cached fastcore-1

In [4]:
%pip install pytorch_tabnet

  Using cached pytorch_tabnet-4.1.0-py3-none-any.whl.metadata (15 kB)
Using cached pytorch_tabnet-4.1.0-py3-none-any.whl (44 kB)

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install syntheval

  Using cached syntheval-1.6.2-py3-none-any.whl.metadata (9.8 kB)
  Using cached pcametric-1.0.4-py3-none-any.whl.metadata (3.8 kB)
Using cached syntheval-1.6.2-py3-none-any.whl (64 kB)
Using cached pcametric-1.0.4-py3-none-any.whl (4.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 74.4 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.12.0
    Uninstalling scipy-1.12.0:
      Successfully uninstalled scipy-1.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [syntheval]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.16.2 which is incompatible.
tabpfn-extensions 0.1.0 requires scikit-learn<1.7,>=1.2.0, but you have scikit-learn 1.7.2 which is incompatible.
smartnoise-synth 1.0.5 requires opacus<0.15.0,>=0.14.0, but you have opacus 1

In [1]:
# from data_synthesizer.util import  plot_training_loss, ModelType
from typing import Tuple
import pandas as pd
import pickle

from data_loader import DataLoader
from data_evaluator import ClassifierType
from data_synthesizer.pipeline import PipelineBuilder, PipelineResults, load_all_results
from data_synthesizer.privacy_sampling import get_epsilon

from evaluation_report.ressemblance_report import ResemblanceReport
from evaluation_report.utility_report import UtilityReport
from evaluation_report.privacy_report import PrivacyReport 
from evaluation_report.privacy_anonymeter_report import PrivacyAnonymeterReport

In [2]:
cat_list_credit_card = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'default.payment.next.month']
num_list_credit_card = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4','BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3','PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
credit_qai_columns = ['LIMIT_BAL','SEX','EDUCATION','MARRIAGE','AGE']
credit_risk_column = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6','default.payment.next.month']

df_real_credit_card_train = DataLoader('../../data/credit_card_Train.csv').get_dataframe(cat_list_credit_card, str, drop_identation=True)
df_real_credit_card_test = DataLoader('../../data/credit_card_Test.csv').get_dataframe(cat_list_credit_card, str, drop_identation=True)
credit_real_dict ={'data' : df_real_credit_card_train, 'cat_list' : cat_list_credit_card, 'num_list' : num_list_credit_card}


In [3]:
cat_list_adult = ['workclass','education','marital-status','occupation','relationship','race','sex','native-country','income']
num_list_adult = ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
adult_qai_columns = ['education','education-num','marital-status','occupation','relationship','race','sex', 'native-country']
adult_risk_column = ['capital-gain','capital-loss','hours-per-week','native-country','income']
df_real_adult_train = DataLoader('../../data/adult_train.csv').get_dataframe(cat_list_adult, str)
df_real_adult_test = DataLoader('../../data/adult_test.csv').get_dataframe(cat_list_adult, str)


In [4]:
num_list_cardio = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']
cat_list_cardio = ['gender','cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']

cardio_qai_columns = ['age','gender','height','weight']
cardio_risk_column = ['ap_lo','ap_hi','cholesterol','gluc','smoke','alco','active','cardio']

df_real_cardio_train = DataLoader('../../data/cardio_train.csv').get_dataframe(cat_list_cardio, category_type=str, sep = ',')
df_real_cardio_test = DataLoader('../../data/cardio_test.csv').get_dataframe(cat_list_cardio, category_type=str, sep = ',')


In [5]:
credit_ctgan  = load_all_results('../results/baseline/credit_ctgan_baseline')
credit_tvae  = load_all_results('../results/baseline/credit_tvae_baseline')



credit_ctgan  = load_all_results('../results/baseline/credit_ctgan_baseline')
credit_tvae  = load_all_results('../results/baseline/credit_tvae_baseline')
credit_ctgan_heom_any_eps_0005  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.005')
credit_ctgan_heom_any_eps_001  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.01')
credit_ctgan_heom_any_eps_005  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.05')
credit_ctgan_heom_any_eps_01  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.1')
credit_ctgan_heom_any_eps_015  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.15')
credit_ctgan_heom_any_eps_02  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.2')
credit_ctgan_heom_any_eps_025  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.25')
credit_ctgan_heom_any_eps_03  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.3')
credit_ctgan_heom_any_eps_035  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.35')
credit_ctgan_heom_any_eps_04  = load_all_results('../results/epsilon_comparison_heom_any/credit_ctgan_eps_0.4')

credit_tvae_heom_any_eps_0005  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.005')
credit_tvae_heom_any_eps_001  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.01')
credit_tvae_heom_any_eps_005  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.05')
credit_tvae_heom_any_eps_01  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.1')
credit_tvae_heom_any_eps_015  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.15')
credit_tvae_heom_any_eps_02  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.2')
credit_tvae_heom_any_eps_025  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.25')
credit_tvae_heom_any_eps_03  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.3')
# credit_tvae_heom_any_eps_035  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.35')
credit_tvae_heom_any_eps_04  = load_all_results('../results/epsilon_comparison_heom_any/credit_tvae_eps_0.4')

dict_results_credit_ctgan_heom_any  = {
    'credit CTGAN' : credit_ctgan,
    'credit CTGAN 0.4' : credit_ctgan_heom_any_eps_04,
    'credit CTGAN 0.35' : credit_ctgan_heom_any_eps_035,
    'credit CTGAN 0.3' : credit_ctgan_heom_any_eps_03,
    'credit CTGAN 0.25' : credit_ctgan_heom_any_eps_025,
    'credit CTGAN 0.2' : credit_ctgan_heom_any_eps_02,
    'credit CTGAN 0.15' : credit_ctgan_heom_any_eps_015,
    'credit CTGAN 0.1' : credit_ctgan_heom_any_eps_01,
    'credit CTGAN 0.05' : credit_ctgan_heom_any_eps_005,
    'credit CTGAN 0.01' : credit_ctgan_heom_any_eps_001,
    'credit CTGAN 0.005' : credit_ctgan_heom_any_eps_0005,    
}
dict_results_credit_tvae_heom_any  = {
    'credit TVAE' : credit_tvae,
    'credit TVAE 0.4' : credit_tvae_heom_any_eps_04,
    # 'credit TVAE 0.35' : credit_tvae_heom_any_eps_035,
    'credit TVAE 0.3' :credit_tvae_heom_any_eps_03,
    'credit TVAE 0.25' :credit_tvae_heom_any_eps_025,
    'credit TVAE 0.2' :credit_tvae_heom_any_eps_02,
    'credit TVAE 0.15' :credit_tvae_heom_any_eps_015,
    'credit TVAE 0.1' :credit_tvae_heom_any_eps_01,
    'credit TVAE 0.05' :credit_tvae_heom_any_eps_005,
    'credit TVAE 0.01' :credit_tvae_heom_any_eps_001,
    'credit TVAE 0.005' :credit_tvae_heom_any_eps_0005
}

In [6]:
adult_ctgan  = load_all_results('../results/baseline/adult_ctgan_baseline')
adult_tvae  = load_all_results('../results/baseline/adult_tvae_baseline')

adult_ctgan_heom_any_eps_0005  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.005')
adult_ctgan_heom_any_eps_001  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.01')
adult_ctgan_heom_any_eps_005  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.05')
adult_ctgan_heom_any_eps_01  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.1')
adult_ctgan_heom_any_eps_015  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.15')
adult_ctgan_heom_any_eps_02  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.2')
adult_ctgan_heom_any_eps_025  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.25')
adult_ctgan_heom_any_eps_03  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.3')
adult_ctgan_heom_any_eps_035  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.35')
adult_ctgan_heom_any_eps_04  = load_all_results('../results/epsilon_comparison_heom_any/adult_ctgan_eps_0.4')

adult_tvae_heom_any_eps_0005  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.005')
adult_tvae_heom_any_eps_001  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.01')
adult_tvae_heom_any_eps_005  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.05')
adult_tvae_heom_any_eps_01  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.1')
adult_tvae_heom_any_eps_015  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.15')
adult_tvae_heom_any_eps_02  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.2')
adult_tvae_heom_any_eps_025  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.25')
adult_tvae_heom_any_eps_03  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.3')
adult_tvae_heom_any_eps_035  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.35')
adult_tvae_heom_any_eps_04  = load_all_results('../results/epsilon_comparison_heom_any/adult_tvae_eps_0.4')

dict_results_adult_ctgan_heom_any  = {
    'adult CTGAN' : adult_ctgan,
    'adult CTGAN 0.4' : adult_ctgan_heom_any_eps_04,
    'adult CTGAN 0.35' : adult_ctgan_heom_any_eps_035,
    'adult CTGAN 0.3' : adult_ctgan_heom_any_eps_03,
    'adult CTGAN 0.25' : adult_ctgan_heom_any_eps_025,
    'adult CTGAN 0.2' : adult_ctgan_heom_any_eps_02,
    'adult CTGAN 0.15' : adult_ctgan_heom_any_eps_015,
    'adult CTGAN 0.1' : adult_ctgan_heom_any_eps_01,
    'adult CTGAN 0.05' : adult_ctgan_heom_any_eps_005,
    'adult CTGAN 0.01' : adult_ctgan_heom_any_eps_001,
    'adult CTGAN 0.005' : adult_ctgan_heom_any_eps_0005,    
}
dict_results_adult_tvae_heom_any  = {
    'adult TVAE' : adult_tvae,
    'adult TVAE 0.4' : adult_tvae_heom_any_eps_04,
    'adult TVAE 0.35' : adult_tvae_heom_any_eps_035,
    'adult TVAE 0.3' :adult_tvae_heom_any_eps_03,
    'adult TVAE 0.25' :adult_tvae_heom_any_eps_025,
    'adult TVAE 0.2' :adult_tvae_heom_any_eps_02,
    'adult TVAE 0.15' :adult_tvae_heom_any_eps_015,
    'adult TVAE 0.1' :adult_tvae_heom_any_eps_01,
    'adult TVAE 0.05' :adult_tvae_heom_any_eps_005,
    'adult TVAE 0.01' :adult_tvae_heom_any_eps_001,
    'adult TVAE 0.005' :adult_tvae_heom_any_eps_0005
}

In [17]:
from synthcity.metrics.eval_privacy import DomiasMIAKDE, IdentifiabilityScore, kAnonymization, lDiversityDistinct      # or LIRAMIA, DeltaPresence, …
from synthcity.plugins.core.dataloader import GenericDataLoader
from synthcity.metrics.eval_attacks import DataLeakageLinear, DataLeakageMLP, DataLeakageXGB

def encode_frame(df):
    out = df.copy()
    for col in out:
        if out[col].dtype == "object" or str(out[col].dtype).startswith("category"):
            out[col] = out[col].astype("category").cat.codes   # -1 = NaN
    return out.astype("float32")            # keeps RAM low

for key, value in dict_results_credit_ctgan_heom_any.items() :

    synth_data = DataLoader(dataset=value['generation_results']['synthetic_data']).get_dataframe(cat_list_credit_card)
    train_data = DataLoader(dataset=df_real_credit_card_train).get_dataframe(cat_list_credit_card)
    val_data = DataLoader(dataset=df_real_credit_card_test).get_dataframe(cat_list_credit_card)
    
    real_enc   = encode_frame(train_data)
    val_enc = encode_frame(val_data)
    synth_enc  = encode_frame(synth_data)

    ldr_real_val = GenericDataLoader(val_enc,
                                     sensitive_features= credit_risk_column,
                                     important_features = credit_qai_columns,)
    ldr_real   = GenericDataLoader(real_enc,
                                   sensitive_features= credit_risk_column,
                                    important_features = credit_qai_columns,)
    ldr_synth  = GenericDataLoader(synth_enc,
                                   sensitive_features= credit_risk_column,
                                     important_features = credit_qai_columns,)
    leak = DataLeakageXGB()
    mkdScore = DomiasMIAKDE()
    idScore = IdentifiabilityScore()
    k_anonym_score = kAnonymization()
    lDiversity = lDiversityDistinct()
    # results_leak = leak.evaluate(X_gt=ldr_real, X_syn=ldr_synth)
    # results_mkdScore = mkdScore.evaluate(X_gt=ldr_real_val, X_train =ldr_real,  X_syn=ldr_synth, synth_val_set=ldr_synth)
    results_idScore = idScore.evaluate(X_gt=ldr_real, X_syn=ldr_synth)
    # results_k_anonym_score = k_anonym_score.evaluate(X_gt=ldr_real, X_syn=ldr_synth)
    results_lDiversity = lDiversity.evaluate(X_gt=ldr_real, X_syn=ldr_synth)

    # print(f"{key} DataLeakageLinear : {results_leak}")
    # print(f"{key} DomiasMIAKDE : {results_mkdScore}")
    print(f"{key} IdentifiabilityScore : {results_idScore}")
    # print(f"{key} kAnonymization : {results_k_anonym_score}")
    # print(f"{key} lDiversityDistinct : {results_lDiversity}")


credit CTGAN IdentifiabilityScore : {'score': 0.06654166666666667, 'score_OC': 0.47470833333333334}
credit CTGAN 0.4 IdentifiabilityScore : {'score': 0.06654166666666667, 'score_OC': 0.47670833333333335}
credit CTGAN 0.35 IdentifiabilityScore : {'score': 0.064125, 'score_OC': 0.477125}
credit CTGAN 0.3 IdentifiabilityScore : {'score': 0.06804166666666667, 'score_OC': 0.477}
credit CTGAN 0.25 IdentifiabilityScore : {'score': 0.066875, 'score_OC': 0.47533333333333333}
credit CTGAN 0.2 IdentifiabilityScore : {'score': 0.06529166666666666, 'score_OC': 0.4765}
credit CTGAN 0.15 IdentifiabilityScore : {'score': 0.06525, 'score_OC': 0.473625}
credit CTGAN 0.1 IdentifiabilityScore : {'score': 0.06595833333333333, 'score_OC': 0.4755}
credit CTGAN 0.05 IdentifiabilityScore : {'score': 0.06683333333333333, 'score_OC': 0.47433333333333333}
credit CTGAN 0.01 IdentifiabilityScore : {'score': 0.0615, 'score_OC': 0.475375}
credit CTGAN 0.005 IdentifiabilityScore : {'score': 0.06375, 'score_OC': 0.4774

In [9]:
print(f"{key} IdentifiabilityScore : {results_idScore}")

credit CTGAN 0.005 IdentifiabilityScore : {'score': 0.06375, 'score_OC': 0.4774583333333333}


In [13]:
from synthcity.metrics.eval_privacy import DomiasMIAKDE, IdentifiabilityScore, kAnonymization, lDiversityDistinct      # or LIRAMIA, DeltaPresence, …
from synthcity.plugins.core.dataloader import GenericDataLoader
from synthcity.metrics.eval_attacks import DataLeakageLinear, DataLeakageMLP, DataLeakageXGB

def encode_frame(df):
    out = df.copy()
    for col in out:
        if out[col].dtype == "object" or str(out[col].dtype).startswith("category"):
            out[col] = out[col].astype("category").cat.codes   # -1 = NaN
    return out.astype("float32")            # keeps RAM low

for key, value in dict_results_adult_ctgan_heom_any.items() :

    synth_data = DataLoader(dataset=value['generation_results']['synthetic_data']).get_dataframe(cat_list_adult)
    train_data = DataLoader(dataset=df_real_adult_train).get_dataframe(cat_list_adult)
    val_data = DataLoader(dataset=df_real_adult_test).get_dataframe(cat_list_adult)
    
    real_enc   = encode_frame(train_data)
    val_enc = encode_frame(val_data)
    synth_enc  = encode_frame(synth_data)

    ldr_real_val = GenericDataLoader(val_enc,
                                     sensitive_features= adult_risk_column,
                                     important_features = adult_qai_columns,)
    ldr_real   = GenericDataLoader(real_enc,
                                   sensitive_features= adult_risk_column,
                                     important_features = adult_qai_columns,)
    ldr_synth  = GenericDataLoader(synth_enc,
                                   sensitive_features= adult_risk_column,
                                     important_features = adult_qai_columns,)
    leak = DataLeakageXGB()
    mkdScore = DomiasMIAKDE()
    idScore = IdentifiabilityScore()
    k_anonym_score = kAnonymization()
    # lDiversity = lDiversityDistinct()
    # results_leak = leak.evaluate(X_gt=ldr_real, X_syn=ldr_synth)
    # results_mkdScore = mkdScore.evaluate(X_gt=ldr_real_val, X_train =ldr_real,  X_syn=ldr_synth, synth_val_set=ldr_synth)
    results_idScore = idScore.evaluate(X_gt=ldr_real, X_syn=ldr_synth)
    results_k_anonym_score = k_anonym_score.evaluate(X_gt=ldr_real, X_syn=ldr_synth)
    results_lDiversity = lDiversity.evaluate(X_gt=ldr_real, X_syn=ldr_synth)

    # print(f"{key} DataLeakageLinear : {results_leak}")
    # print(f"{key} DomiasMIAKDE : {results_mkdScore}")
    print(f"{key} IdentifiabilityScore : {results_idScore}")
    print(f"{key} kAnonymization : {results_k_anonym_score}")
    # print(f"{key} lDiversityDistinct : {results_lDiversity}")

adult CTGAN IdentifiabilityScore : {'score': 0.26937133380424433, 'score_OC': 0.4748932772335002}
adult CTGAN kAnonymization : {'gt': 20, 'syn': 136.00000001}
adult CTGAN 0.4 IdentifiabilityScore : {'score': 0.26737508061791715, 'score_OC': 0.4820490771167962}
adult CTGAN 0.4 kAnonymization : {'gt': 20, 'syn': 133.00000001}
adult CTGAN 0.35 IdentifiabilityScore : {'score': 0.26430392186972146, 'score_OC': 0.48072847885507203}
adult CTGAN 0.35 kAnonymization : {'gt': 20, 'syn': 88.00000001}
adult CTGAN 0.3 IdentifiabilityScore : {'score': 0.2678664660176284, 'score_OC': 0.47928503424342006}
adult CTGAN 0.3 kAnonymization : {'gt': 20, 'syn': 159.00000001}
adult CTGAN 0.25 IdentifiabilityScore : {'score': 0.26614661711863885, 'score_OC': 0.4759067596204048}
adult CTGAN 0.25 kAnonymization : {'gt': 20, 'syn': 109.00000001}
adult CTGAN 0.2 IdentifiabilityScore : {'score': 0.26488744203187864, 'score_OC': 0.48164982647953075}
adult CTGAN 0.2 kAnonymization : {'gt': 20, 'syn': 207.00000001}
a

In [14]:
from syntheval import SynthEval

evaluator = SynthEval(df_real_credit_card_train, holdout_dataframe = df_real_credit_card_test, cat_cols = cat_list_credit_card)

In [7]:
metrics = {
    # "cio"       : {"confidence": 95},
    # "corr_diff" : {"mixed_corr": True},
    # "mi_diff"   : {},
    "ks_test"   : {"sig_lvl": 0.05, "n_perms": 1000},
    # "h_dist"    : {},
    # "p_mse"     : {"k_folds": 5, "max_iter": 100, "solver": "liblinear"},
    # "nnaa"      : {"n_resample": 30},
    # "cls_acc"   : {"F1_type": "micro", "k_folds": 5},
    # "hit_rate"  : {"thres_percent": 0.0333},
    # "eps_risk"  : {},
    # "mia_risk"  : {"num_eval_iter": 5}
}

In [18]:
privacy_report = evaluator.evaluate(
    synth_data,
    class_lab_col=None,        # supply a target column only if you need MIA class attack colours
    presets_file="privacy",    # "full_eval" | "fast_eval" | "privacy"

)

SynthEval: synthetic data read successfully


Syntheval: class_lab_col: 100%|██████████| 9/9 [26:36<00:00, 177.42s/it]

Unrecognised keyword: class_lab_col

SynthEval results

Utility metric description                    value   error                                 
+---------------------------------------------------------------+
| Propensity mean squared error (pMSE)     :   0.0184  0.0002   |
|   -> average pMSE classifier accuracy    :   0.5978  0.0026   |
| Nearest neighbour adversarial accuracy   :   0.7906  0.0000   |
+---------------------------------------------------------------+
    
Privacy metric description                    value   error                                 
+---------------------------------------------------------------+
| Nearest neighbour distance ratio         :   0.8402  0.0011   |
| Privacy loss (diff. in NNDR)             :   -0.0213  0.0027   |
| Privacy loss (diff. in NNAA)             :   -0.1260  0.0009   |
| Median distance to closest record        :   7.2754           |
| Hitting rate (0.03 x range(att))         :   0.2009           |
| Epsilon identifiability